
# Multiperiod Arbitrage-Consistent Projection (ACP)

This Colab notebook implements **multiperiod ACP** under the physical-settlement / passive-carry convention.

It contains two ACP solvers:

```python
solve_multiperiod_acp_bid_ask(...)
solve_multiperiod_acp_frictionless(...)
```

The bid-ask solver is the main practical routine. It does **not** force equality to bid and ask quotes. Instead, it constructs strictly positive dual weights and shadow prices satisfying

\[
q^{bid}_{corrected} \le \langle \ell, B_W\rangle \le q^{ask}_{corrected}.
\]

That is the correct bid-ask ACP logic: the shadow price must lie inside the corrected executable spread.



## Cell 1 - Install packages

Run this cell first in Colab.


In [ ]:
%pip -q install numpy pandas scipy matplotlib


## Cell 2 - Full implementation

This cell defines the physical-settlement payoff map \(B_{phys}\), the bid-ask ACP LP, and the frictionless ACP LP.

The comments explain how the finite exercise-pattern construction is translated into code.


In [ ]:

# ============================================================
# Multiperiod Arbitrage-Consistent Projection (ACP)
# Physical settlement + passive carry, with bid-ask shadow prices
# ============================================================
#
# This code implements a multiperiod version of the Arbitrage-Consistent
# Projection idea from the paper:
#
#   "Deterministic Arbitrage, Localized Arbitrage Portfolios,
#    and Arbitrage-Consistent Projection"
#
# The notebook is designed for Colab and is intentionally written as a
# self-contained script.
#
# Financial convention
# --------------------
# The multi-maturity convention is physical settlement with passive carry:
#
# 1. All positions are chosen at time zero.
# 2. There is no discretionary rebalancing at intermediate maturities.
# 3. If an option matures in the money before the final horizon, its
#    physical settlement mechanically creates a stock-and-cash position.
# 4. That stock-and-cash position is then carried passively to the final
#    horizon.
# 5. Options with zero intrinsic value are assumed not to be physically
#    settled:
#
#       call settles only when S_Tj > K,
#       put  settles only when S_Tj < K.
#
# ACP idea
# --------
# The detector searches directly for arbitrage portfolios. ACP works in
# the opposite direction: it searches for corrected prices that admit a
# strictly positive dual certificate.
#
# In the frictionless case, the certificate is:
#
#       B_phys.T @ ell = q_corrected,
#       ell > 0.
#
# In the bid-ask case, equality to executable bid and ask prices is wrong,
# because it would collapse the spread. The correct condition is a shadow
# price interval:
#
#       bid_corrected <= <ell, payoff_column> <= ask_corrected.
#
# The code below provides:
#
#   solve_multiperiod_acp_bid_ask(...)
#       Corrects a full bid-ask option book by minimum weighted L1 change.
#
#   solve_multiperiod_acp_frictionless(...)
#       Corrects one price per call/put by minimum weighted L1 change.
#
# The bid-ask ACP is the main practical routine. The frictionless routine
# is included because it is useful for testing and for model-generated
# single-price forecasts.

import itertools
import math
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.optimize import linprog

try:
    from IPython.display import display
except Exception:
    display = print


# ------------------------------------------------------------
# 1. Input cleaning
# ------------------------------------------------------------

REQUIRED_BID_ASK_COLUMNS = [
    "T", "K",
    "call_bid", "call_ask",
    "put_bid", "put_ask",
]

REQUIRED_FRICTIONLESS_COLUMNS = [
    "T", "K", "call", "put"
]


def clean_bid_ask_quotes(quotes: pd.DataFrame) -> pd.DataFrame:
    """Validate and sort a multi-maturity bid-ask option quote table."""
    missing = [c for c in REQUIRED_BID_ASK_COLUMNS if c not in quotes.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = quotes[REQUIRED_BID_ASK_COLUMNS].copy()
    for col in REQUIRED_BID_ASK_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="raise")

    if (df["T"] <= 0).any():
        raise ValueError("All maturities T must be strictly positive.")
    if (df["K"] <= 0).any():
        raise ValueError("All strikes K must be strictly positive.")
    if (df["call_bid"] > df["call_ask"]).any():
        raise ValueError("Found call_bid > call_ask.")
    if (df["put_bid"] > df["put_ask"]).any():
        raise ValueError("Found put_bid > put_ask.")
    if df.duplicated(["T", "K"]).any():
        raise ValueError("Duplicate (T, K) rows are not allowed.")

    return df.sort_values(["T", "K"]).reset_index(drop=True)


def clean_frictionless_quotes(quotes: pd.DataFrame) -> pd.DataFrame:
    """Validate and sort a multi-maturity single-price option quote table."""
    missing = [c for c in REQUIRED_FRICTIONLESS_COLUMNS if c not in quotes.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = quotes[REQUIRED_FRICTIONLESS_COLUMNS].copy()
    for col in REQUIRED_FRICTIONLESS_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="raise")

    if (df["T"] <= 0).any():
        raise ValueError("All maturities T must be strictly positive.")
    if (df["K"] <= 0).any():
        raise ValueError("All strikes K must be strictly positive.")
    if df.duplicated(["T", "K"]).any():
        raise ValueError("Duplicate (T, K) rows are not allowed.")

    return df.sort_values(["T", "K"]).reset_index(drop=True)


def mid_quotes_from_bid_ask(quotes_bid_ask: pd.DataFrame) -> pd.DataFrame:
    """Create a frictionless/mid quote table from bid-ask data."""
    df = clean_bid_ask_quotes(quotes_bid_ask)
    return pd.DataFrame({
        "T": df["T"],
        "K": df["K"],
        "call": 0.5 * (df["call_bid"] + df["call_ask"]),
        "put": 0.5 * (df["put_bid"] + df["put_ask"]),
    })


def build_book_from_bid_ask(quotes: pd.DataFrame, S0: float, r: float) -> Dict[str, Any]:
    """Build the internal market representation for bid-ask ACP."""
    df = clean_bid_ask_quotes(quotes)
    S0 = float(S0)
    r = float(r)
    if S0 <= 0:
        raise ValueError("S0 must be strictly positive.")

    maturities = sorted(df["T"].unique().astype(float))
    strikes = []
    call_bid = []
    call_ask = []
    put_bid = []
    put_ask = []

    for T in maturities:
        g = df[df["T"] == T].sort_values("K")
        strikes.append(g["K"].to_numpy(float))
        call_bid.append(g["call_bid"].to_numpy(float))
        call_ask.append(g["call_ask"].to_numpy(float))
        put_bid.append(g["put_bid"].to_numpy(float))
        put_ask.append(g["put_ask"].to_numpy(float))

    option_records = []
    for j, T in enumerate(maturities):
        for i, K in enumerate(strikes[j]):
            option_records.append({
                "kind": "call",
                "j": j,
                "i": i,
                "T": float(T),
                "K": float(K),
                "raw_bid": float(call_bid[j][i]),
                "raw_ask": float(call_ask[j][i]),
                "name": f"C(T={T:g}, K={K:g})",
            })
            option_records.append({
                "kind": "put",
                "j": j,
                "i": i,
                "T": float(T),
                "K": float(K),
                "raw_bid": float(put_bid[j][i]),
                "raw_ask": float(put_ask[j][i]),
                "name": f"P(T={T:g}, K={K:g})",
            })

    return {
        "quotes": df,
        "S0": S0,
        "r": r,
        "maturities": maturities,
        "m": len(maturities),
        "T_final": maturities[-1],
        "strikes": strikes,
        "option_records": option_records,
    }


def build_book_from_frictionless(quotes: pd.DataFrame, S0: float, r: float) -> Dict[str, Any]:
    """Build the internal market representation for frictionless ACP."""
    df = clean_frictionless_quotes(quotes)
    S0 = float(S0)
    r = float(r)
    if S0 <= 0:
        raise ValueError("S0 must be strictly positive.")

    maturities = sorted(df["T"].unique().astype(float))
    strikes = []
    call = []
    put = []

    for T in maturities:
        g = df[df["T"] == T].sort_values("K")
        strikes.append(g["K"].to_numpy(float))
        call.append(g["call"].to_numpy(float))
        put.append(g["put"].to_numpy(float))

    option_records = []
    for j, T in enumerate(maturities):
        for i, K in enumerate(strikes[j]):
            option_records.append({
                "kind": "call",
                "j": j,
                "i": i,
                "T": float(T),
                "K": float(K),
                "raw_price": float(call[j][i]),
                "name": f"C(T={T:g}, K={K:g})",
            })
            option_records.append({
                "kind": "put",
                "j": j,
                "i": i,
                "T": float(T),
                "K": float(K),
                "raw_price": float(put[j][i]),
                "name": f"P(T={T:g}, K={K:g})",
            })

    return {
        "quotes": df,
        "S0": S0,
        "r": r,
        "maturities": maturities,
        "m": len(maturities),
        "T_final": maturities[-1],
        "strikes": strikes,
        "option_records": option_records,
    }


# ------------------------------------------------------------
# 2. Exercise patterns and physical-settlement payoff map B_phys
# ------------------------------------------------------------

def enumerate_exercise_patterns_for_strikes(strikes: np.ndarray, maturity_label: str = ""):
    """Enumerate all reachable exercise patterns at one pre-final maturity."""
    K = np.asarray(strikes, dtype=float)
    n = len(K)

    if n == 0:
        return [{
            "label": f"{maturity_label}: no strikes",
            "representative": np.nan,
            "call": np.zeros(0),
            "put": np.zeros(0),
        }]

    reps_and_labels = [(0.0, f"{maturity_label}: s < {K[0]:g}")]

    for i, k in enumerate(K):
        reps_and_labels.append((float(k), f"{maturity_label}: s = {k:g}"))
        if i < n - 1:
            mid = 0.5 * (K[i] + K[i + 1])
            reps_and_labels.append((float(mid), f"{maturity_label}: {K[i]:g} < s < {K[i+1]:g}"))

    above = float(K[-1] + max(1.0, 0.10 * abs(K[-1])))
    reps_and_labels.append((above, f"{maturity_label}: s > {K[-1]:g}"))

    patterns = []
    for rep, label in reps_and_labels:
        patterns.append({
            "label": label,
            "representative": rep,
            "call": (rep > K).astype(float),
            "put": (rep < K).astype(float),
        })

    return patterns


def enumerate_prefinal_patterns(book: Dict[str, Any]):
    """Build the Cartesian product of all pre-final exercise patterns."""
    m = book["m"]
    if m == 1:
        return [{"label": "no pre-final maturities", "components": []}]

    per_maturity = []
    for j in range(m - 1):
        label = f"T{j+1}={book['maturities'][j]:g}"
        per_maturity.append(enumerate_exercise_patterns_for_strikes(book["strikes"][j], label))

    all_patterns = []
    for combo in itertools.product(*per_maturity):
        all_patterns.append({
            "label": " | ".join(p["label"] for p in combo),
            "components": list(combo),
        })

    return all_patterns


def final_certificate_points(book: Dict[str, Any]) -> np.ndarray:
    """Endpoint/breakpoint set for the final maturity: {0, final strikes}."""
    final_strikes = np.asarray(book["strikes"][-1], dtype=float)
    return np.sort(np.unique(np.concatenate([np.array([0.0]), final_strikes])))


def option_terminal_coefficient(book, option, pattern, x: float) -> float:
    """Terminal wealth column value for one unit of an option position."""
    j = option["j"]
    K = option["K"]
    Tj = option["T"]
    T_final = book["T_final"]
    r = book["r"]

    if j == book["m"] - 1:
        if option["kind"] == "call":
            return max(float(x) - K, 0.0)
        else:
            return max(K - float(x), 0.0)

    A = math.exp(r * (T_final - Tj))
    component = pattern["components"][j]
    i = option["i"]

    if option["kind"] == "call":
        c = component["call"][i]
        return float(c) * (float(x) - K * A)
    else:
        p = component["put"][i]
        return float(p) * (K * A - float(x))


def option_tail_slope_coefficient(book, option, pattern) -> float:
    """Right-tail slope column value for one unit of an option position."""
    j = option["j"]

    if j == book["m"] - 1:
        return 1.0 if option["kind"] == "call" else 0.0

    component = pattern["components"][j]
    i = option["i"]

    if option["kind"] == "call":
        return float(component["call"][i])
    else:
        return -float(component["put"][i])


def build_physical_settlement_payoff_map(book: Dict[str, Any]) -> Dict[str, Any]:
    """Build B_phys, the finite physical-settlement payoff map.

    Columns are instruments:
        cash, stock, all calls, all puts.

    Rows are:
        G^pattern(x) for each pre-final pattern and each final certificate
        point x in {0, final strikes}, plus one tail-slope row for each
        pattern.

    A nonnegative vector B_phys @ theta means that the portfolio terminal
    wealth is nonnegative for every path, because each patternwise payoff
    is piecewise affine in the final stock price and has nonnegative tail
    slope.
    """
    patterns = enumerate_prefinal_patterns(book)
    x_points = final_certificate_points(book)

    instruments = [
        {"kind": "cash", "name": "cash"},
        {"kind": "stock", "name": "stock"},
    ] + [dict(opt) for opt in book["option_records"]]

    rows = []
    row_meta = []

    exp_rT = math.exp(book["r"] * book["T_final"])

    for p_idx, pat in enumerate(patterns):
        for x in x_points:
            row = []
            for instr in instruments:
                if instr["kind"] == "cash":
                    row.append(exp_rT)
                elif instr["kind"] == "stock":
                    row.append(float(x))
                else:
                    row.append(option_terminal_coefficient(book, instr, pat, float(x)))
            rows.append(row)
            row_meta.append({
                "row_type": "value",
                "pattern_index": p_idx,
                "x": float(x),
                "pattern": pat["label"],
            })

        # Tail slope row.
        row = []
        for instr in instruments:
            if instr["kind"] == "cash":
                row.append(0.0)
            elif instr["kind"] == "stock":
                row.append(1.0)
            else:
                row.append(option_tail_slope_coefficient(book, instr, pat))
        rows.append(row)
        row_meta.append({
            "row_type": "tail_slope",
            "pattern_index": p_idx,
            "x": np.inf,
            "pattern": pat["label"],
        })

    B = np.asarray(rows, dtype=float)

    return {
        "B": B,
        "instruments": instruments,
        "patterns": patterns,
        "x_points": x_points,
        "row_meta": pd.DataFrame(row_meta),
    }


# ------------------------------------------------------------
# 3. Bid-ask ACP
# ------------------------------------------------------------

def solve_multiperiod_acp_bid_ask(
    quotes: pd.DataFrame,
    S0: float,
    r: float,
    ell_lower: float = 1e-10,
    quote_floor: float = 0.0,
    weights: Optional[Dict[str, float]] = None,
    solver_options: Optional[Dict[str, Any]] = None,
):
    """Solve the multiperiod bid-ask ACP linear programme.

    Parameters
    ----------
    quotes:
        DataFrame with columns:
            T, K, call_bid, call_ask, put_bid, put_ask
    S0, r:
        Current stock price and continuously compounded risk-free rate.
    ell_lower:
        Strict-positivity tolerance for the dual certificate. Use a small
        positive number. If it is too large, the LP may become infeasible
        for purely numerical reasons.
    quote_floor:
        Lower bound for corrected option bid/ask quotes. Usually 0.
    weights:
        Optional L1 weights for quote-side corrections. Keys:
            call_bid, call_ask, put_bid, put_ask
        Missing keys default to 1.
    solver_options:
        Optional scipy.optimize.linprog options.

    Returns
    -------
    result, model:
        The SciPy result and a dictionary with all indexing information.

    Correct bid-ask logic
    ---------------------
    For every option payoff column B_W, the certificate defines a shadow
    price p_W = <ell, B_W>. The corrected quotes must satisfy:

        W_bid_corrected <= p_W <= W_ask_corrected.

    This is the correct bid-ask condition. Do NOT impose equality to both
    bid and ask; that would force bid = ask.
    """
    if weights is None:
        weights = {}
    default_weights = {
        "call_bid": 1.0,
        "call_ask": 1.0,
        "put_bid": 1.0,
        "put_ask": 1.0,
    }
    default_weights.update(weights)
    weights = default_weights

    book = build_book_from_bid_ask(quotes, S0=S0, r=r)
    fmap = build_physical_settlement_payoff_map(book)

    B = fmap["B"]
    instruments = fmap["instruments"]
    n_rows, n_instr = B.shape

    # Instrument column indices.
    cash_col = 0
    stock_col = 1

    # Variable creation helper.
    var_names = []
    bounds = []

    def add_var(name, lb=0.0, ub=None):
        idx = len(var_names)
        var_names.append(name)
        bounds.append((lb, ub))
        return idx

    # Dual certificate variables ell_r, one per finite payoff-map row.
    ell_idx = [add_var(f"ell[{ridx}]", lb=ell_lower, ub=None) for ridx in range(n_rows)]

    # Corrected quote and absolute-deviation variables.
    quote_idx = {}
    eps_plus_idx = {}
    eps_minus_idx = {}

    option_columns = []
    for col, instr in enumerate(instruments):
        if instr["kind"] not in ("call", "put"):
            continue

        option_columns.append(col)
        for side in ("bid", "ask"):
            qidx = add_var(f"{instr['name']} corrected_{side}", lb=quote_floor, ub=None)
            ep = add_var(f"{instr['name']} {side} adjustment_plus", lb=0.0, ub=None)
            em = add_var(f"{instr['name']} {side} adjustment_minus", lb=0.0, ub=None)

            key = (instr["kind"], instr["j"], instr["i"], side)
            quote_idx[key] = qidx
            eps_plus_idx[key] = ep
            eps_minus_idx[key] = em

    nvar = len(var_names)

    c = np.zeros(nvar)
    for col in option_columns:
        instr = instruments[col]
        for side in ("bid", "ask"):
            key = (instr["kind"], instr["j"], instr["i"], side)
            weight_key = f"{instr['kind']}_{side}"
            c[eps_plus_idx[key]] = weights[weight_key]
            c[eps_minus_idx[key]] = weights[weight_key]

    A_eq = []
    b_eq = []

    def ell_dot_column_row(col):
        row = np.zeros(nvar)
        for ridx in range(n_rows):
            row[ell_idx[ridx]] = B[ridx, col]
        return row

    # Cash and stock are treated as frictionless base instruments:
    #     <ell, cash column> = 1,
    #     <ell, stock column> = S0.
    row = ell_dot_column_row(cash_col)
    A_eq.append(row)
    b_eq.append(1.0)

    row = ell_dot_column_row(stock_col)
    A_eq.append(row)
    b_eq.append(float(S0))

    # Corrected quote definitions:
    #     q_corrected = q_raw + eps_plus - eps_minus
    # written as
    #     q_corrected - eps_plus + eps_minus = q_raw.
    for col in option_columns:
        instr = instruments[col]
        for side in ("bid", "ask"):
            key = (instr["kind"], instr["j"], instr["i"], side)
            row = np.zeros(nvar)
            row[quote_idx[key]] = 1.0
            row[eps_plus_idx[key]] = -1.0
            row[eps_minus_idx[key]] = 1.0
            A_eq.append(row)
            b_eq.append(instr[f"raw_{side}"])

    A_ub = []
    b_ub = []

    # Shadow-price interval and spread constraints.
    for col in option_columns:
        instr = instruments[col]
        key_bid = (instr["kind"], instr["j"], instr["i"], "bid")
        key_ask = (instr["kind"], instr["j"], instr["i"], "ask")

        # corrected_bid <= shadow
        # corrected_bid - <ell, B_col> <= 0
        row = np.zeros(nvar)
        row[quote_idx[key_bid]] = 1.0
        for ridx in range(n_rows):
            row[ell_idx[ridx]] -= B[ridx, col]
        A_ub.append(row)
        b_ub.append(0.0)

        # shadow <= corrected_ask
        # <ell, B_col> - corrected_ask <= 0
        row = np.zeros(nvar)
        row[quote_idx[key_ask]] = -1.0
        for ridx in range(n_rows):
            row[ell_idx[ridx]] += B[ridx, col]
        A_ub.append(row)
        b_ub.append(0.0)

        # corrected_bid <= corrected_ask
        row = np.zeros(nvar)
        row[quote_idx[key_bid]] = 1.0
        row[quote_idx[key_ask]] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    if solver_options is None:
        solver_options = {}

    result = linprog(
        c=c,
        A_ub=np.asarray(A_ub),
        b_ub=np.asarray(b_ub),
        A_eq=np.asarray(A_eq),
        b_eq=np.asarray(b_eq),
        bounds=bounds,
        method="highs",
        options=solver_options,
    )

    model = {
        "mode": "bid_ask",
        "book": book,
        "payoff_map": fmap,
        "B": B,
        "instruments": instruments,
        "cash_col": cash_col,
        "stock_col": stock_col,
        "option_columns": option_columns,
        "var_names": var_names,
        "bounds": bounds,
        "ell_idx": ell_idx,
        "quote_idx": quote_idx,
        "eps_plus_idx": eps_plus_idx,
        "eps_minus_idx": eps_minus_idx,
        "objective_vector": c,
        "A_eq": np.asarray(A_eq),
        "b_eq": np.asarray(b_eq),
        "A_ub": np.asarray(A_ub),
        "b_ub": np.asarray(b_ub),
        "ell_lower": ell_lower,
        "quote_floor": quote_floor,
    }

    return result, model


def extract_bid_ask_acp_solution(result, model: Dict[str, Any]) -> Dict[str, Any]:
    """Extract corrected bid-ask quotes and shadow prices."""
    if not result.success:
        raise RuntimeError(f"ACP LP failed: {result.message}")

    z = result.x
    B = model["B"]
    instruments = model["instruments"]

    ell = np.array([z[i] for i in model["ell_idx"]])
    shadow_all = B.T @ ell

    rows = []
    by_key = {}

    # First store all raw/corrected values by (j, i).
    for col in model["option_columns"]:
        instr = instruments[col]
        kind = instr["kind"]
        j = instr["j"]
        i = instr["i"]
        base_key = (j, i)

        if base_key not in by_key:
            by_key[base_key] = {
                "T": instr["T"],
                "K": instr["K"],
            }

        for side in ("bid", "ask"):
            key = (kind, j, i, side)
            by_key[base_key][f"{kind}_{side}_raw"] = instr[f"raw_{side}"]
            by_key[base_key][f"{kind}_{side}_acp"] = z[model["quote_idx"][key]]

        by_key[base_key][f"{kind}_shadow"] = shadow_all[col]

    for key in sorted(by_key.keys()):
        rows.append(by_key[key])

    corrected = pd.DataFrame(rows).sort_values(["T", "K"]).reset_index(drop=True)

    total_l1 = float(model["objective_vector"] @ z)

    return {
        "ell": ell,
        "shadow_all": shadow_all,
        "corrected_quotes": corrected,
        "objective_value": total_l1,
    }


def print_bid_ask_acp_report(result, model: Dict[str, Any], rows: int = 20):
    """Print a readable ACP report."""
    print("Solver status:", result.message)
    if not result.success:
        return

    sol = extract_bid_ask_acp_solution(result, model)
    print(f"Minimum weighted L1 correction: {sol['objective_value']:.12g}")
    print(f"Minimum ell component: {sol['ell'].min():.12g}")
    print(f"Number of dual certificate rows: {len(sol['ell'])}")

    print("\nCorrected bid-ask quotes and shadow prices:")
    display(sol["corrected_quotes"].head(rows))

    # Small numerical checks for the shadow intervals.
    df = sol["corrected_quotes"]
    checks = []
    for _, row in df.iterrows():
        checks.append(row["call_bid_acp"] <= row["call_shadow"] + 1e-8)
        checks.append(row["call_shadow"] <= row["call_ask_acp"] + 1e-8)
        checks.append(row["put_bid_acp"] <= row["put_shadow"] + 1e-8)
        checks.append(row["put_shadow"] <= row["put_ask_acp"] + 1e-8)
    print("\nAll shadow prices inside corrected bid-ask intervals:",
          bool(np.all(checks)))


# ------------------------------------------------------------
# 4. Frictionless ACP
# ------------------------------------------------------------

def solve_multiperiod_acp_frictionless(
    quotes: pd.DataFrame,
    S0: float,
    r: float,
    ell_lower: float = 1e-10,
    weights: Optional[Dict[str, float]] = None,
    solver_options: Optional[Dict[str, Any]] = None,
):
    """Solve the multiperiod frictionless ACP linear programme.

    This routine corrects one call price and one put price per row.
    It is useful when the raw values are model forecasts rather than
    executable bid-ask quotes.

    For each option payoff column B_W:

        <ell, B_W> = raw_price + eps_plus - eps_minus.

    The objective minimizes the weighted sum of eps_plus + eps_minus.
    """
    if weights is None:
        weights = {}
    weights = {"call": weights.get("call", 1.0), "put": weights.get("put", 1.0)}

    book = build_book_from_frictionless(quotes, S0=S0, r=r)
    fmap = build_physical_settlement_payoff_map(book)

    B = fmap["B"]
    instruments = fmap["instruments"]
    n_rows, _n_instr = B.shape

    cash_col = 0
    stock_col = 1

    var_names = []
    bounds = []

    def add_var(name, lb=0.0, ub=None):
        idx = len(var_names)
        var_names.append(name)
        bounds.append((lb, ub))
        return idx

    ell_idx = [add_var(f"ell[{ridx}]", lb=ell_lower, ub=None) for ridx in range(n_rows)]

    eps_plus_idx = {}
    eps_minus_idx = {}

    option_columns = []
    for col, instr in enumerate(instruments):
        if instr["kind"] not in ("call", "put"):
            continue
        option_columns.append(col)
        key = (instr["kind"], instr["j"], instr["i"])
        eps_plus_idx[key] = add_var(f"{instr['name']} adjustment_plus", lb=0.0, ub=None)
        eps_minus_idx[key] = add_var(f"{instr['name']} adjustment_minus", lb=0.0, ub=None)

    nvar = len(var_names)

    c = np.zeros(nvar)
    for col in option_columns:
        instr = instruments[col]
        key = (instr["kind"], instr["j"], instr["i"])
        c[eps_plus_idx[key]] = weights[instr["kind"]]
        c[eps_minus_idx[key]] = weights[instr["kind"]]

    A_eq = []
    b_eq = []

    def ell_dot_column_row(col):
        row = np.zeros(nvar)
        for ridx in range(n_rows):
            row[ell_idx[ridx]] = B[ridx, col]
        return row

    # Frictionless cash and stock pricing.
    row = ell_dot_column_row(cash_col)
    A_eq.append(row)
    b_eq.append(1.0)

    row = ell_dot_column_row(stock_col)
    A_eq.append(row)
    b_eq.append(float(S0))

    # Option pricing equations:
    #     <ell, B_col> = raw + eps_plus - eps_minus
    # becomes
    #     <ell, B_col> - eps_plus + eps_minus = raw.
    for col in option_columns:
        instr = instruments[col]
        key = (instr["kind"], instr["j"], instr["i"])
        row = ell_dot_column_row(col)
        row[eps_plus_idx[key]] = -1.0
        row[eps_minus_idx[key]] = 1.0
        A_eq.append(row)
        b_eq.append(instr["raw_price"])

    if solver_options is None:
        solver_options = {}

    result = linprog(
        c=c,
        A_eq=np.asarray(A_eq),
        b_eq=np.asarray(b_eq),
        bounds=bounds,
        method="highs",
        options=solver_options,
    )

    model = {
        "mode": "frictionless",
        "book": book,
        "payoff_map": fmap,
        "B": B,
        "instruments": instruments,
        "cash_col": cash_col,
        "stock_col": stock_col,
        "option_columns": option_columns,
        "var_names": var_names,
        "bounds": bounds,
        "ell_idx": ell_idx,
        "eps_plus_idx": eps_plus_idx,
        "eps_minus_idx": eps_minus_idx,
        "objective_vector": c,
        "ell_lower": ell_lower,
    }

    return result, model


def extract_frictionless_acp_solution(result, model: Dict[str, Any]) -> Dict[str, Any]:
    """Extract corrected frictionless prices from the ACP solution."""
    if not result.success:
        raise RuntimeError(f"ACP LP failed: {result.message}")

    z = result.x
    B = model["B"]
    instruments = model["instruments"]

    ell = np.array([z[i] for i in model["ell_idx"]])
    shadow_all = B.T @ ell

    rows_by_key = {}
    for col in model["option_columns"]:
        instr = instruments[col]
        kind = instr["kind"]
        j = instr["j"]
        i = instr["i"]
        base_key = (j, i)

        if base_key not in rows_by_key:
            rows_by_key[base_key] = {
                "T": instr["T"],
                "K": instr["K"],
            }

        key = (kind, j, i)
        ep = z[model["eps_plus_idx"][key]]
        em = z[model["eps_minus_idx"][key]]
        raw = instr["raw_price"]
        corrected = raw + ep - em

        rows_by_key[base_key][f"{kind}_raw"] = raw
        rows_by_key[base_key][f"{kind}_acp"] = corrected
        rows_by_key[base_key][f"{kind}_shadow"] = shadow_all[col]
        rows_by_key[base_key][f"{kind}_adjustment"] = corrected - raw

    corrected = pd.DataFrame(
        [rows_by_key[k] for k in sorted(rows_by_key.keys())]
    ).sort_values(["T", "K"]).reset_index(drop=True)

    total_l1 = float(model["objective_vector"] @ z)

    return {
        "ell": ell,
        "shadow_all": shadow_all,
        "corrected_prices": corrected,
        "objective_value": total_l1,
    }


def print_frictionless_acp_report(result, model: Dict[str, Any], rows: int = 20):
    """Print a readable report for frictionless ACP."""
    print("Solver status:", result.message)
    if not result.success:
        return

    sol = extract_frictionless_acp_solution(result, model)
    print(f"Minimum weighted L1 correction: {sol['objective_value']:.12g}")
    print(f"Minimum ell component: {sol['ell'].min():.12g}")
    print(f"Number of dual certificate rows: {len(sol['ell'])}")

    print("\nCorrected prices:")
    display(sol["corrected_prices"].head(rows))


# ------------------------------------------------------------
# 5. Demo data
# ------------------------------------------------------------

def demo_bid_ask_calendar_violation_quotes() -> pd.DataFrame:
    """Small bid-ask book with an executable call-calendar violation.

    The violation is:

        C_bid(T1, K=100) > C_ask(T2, K=100),

    so buying the longer call at ask and selling the shorter call at bid
    has negative initial cost before considering terminal wealth.
    """
    return pd.DataFrame({
        "T":        [0.5,   1.0],
        "K":        [100.0, 100.0],
        "call_bid": [11.9,  10.9],
        "call_ask": [12.1,  11.1],
        # Puts are deliberately wide here. The example is meant to
        # illustrate multi-maturity correction, not put-specific repair.
        "put_bid":  [8.0,   5.0],
        "put_ask":  [15.0,  15.0],
    })


def demo_frictionless_calendar_violation_quotes() -> pd.DataFrame:
    """Single-price version of the same calendar-violation idea."""
    return pd.DataFrame({
        "T":    [0.5,   1.0],
        "K":    [100.0, 100.0],
        "call": [12.0,  11.0],
        "put":  [11.0,  9.5],
    })



## Cell 3 - Demo: bid-ask multiperiod ACP

The demo quote book contains an executable multi-maturity inconsistency. ACP minimally adjusts the bid-ask quotes so that every option payoff has a shadow price inside its corrected bid-ask interval.


In [ ]:

S0_demo = 100.0
r_demo = 0.02

quotes_bid_ask_demo = demo_bid_ask_calendar_violation_quotes()
display(quotes_bid_ask_demo)

result_ba, model_ba = solve_multiperiod_acp_bid_ask(
    quotes=quotes_bid_ask_demo,
    S0=S0_demo,
    r=r_demo,
    ell_lower=1e-10,
    quote_floor=0.0,
)

print_bid_ask_acp_report(result_ba, model_ba)



## Cell 4 - Demo: frictionless multiperiod ACP

This version corrects one price per call and put. It is useful when raw quotes are model forecasts or mid prices rather than executable bid-ask quotes.


In [ ]:

quotes_frictionless_demo = demo_frictionless_calendar_violation_quotes()
display(quotes_frictionless_demo)

result_fr, model_fr = solve_multiperiod_acp_frictionless(
    quotes=quotes_frictionless_demo,
    S0=S0_demo,
    r=r_demo,
    ell_lower=1e-10,
)

print_frictionless_acp_report(result_fr, model_fr)



## Cell 5 - Replace with your own data

For bid-ask ACP, provide:

```text
T, K, call_bid, call_ask, put_bid, put_ask
```

For frictionless ACP, provide:

```text
T, K, call, put
```

Keep `ell_lower` small. If it is too large, the strict interior certificate can become infeasible for numerical rather than financial reasons.


In [ ]:

# -------------------------
# Bid-ask ACP custom template
# -------------------------

my_bid_ask_quotes = pd.DataFrame({
    "T":        [0.25, 0.25, 0.50, 0.50],
    "K":        [95.0, 105.0, 95.0, 105.0],
    "call_bid": [8.0,  3.0,   8.5,  3.5],
    "call_ask": [8.4,  3.4,   8.9,  3.9],
    "put_bid":  [2.5,  7.0,   3.0,  7.5],
    "put_ask":  [2.9,  7.4,   3.4,  7.9],
})

S0 = 100.0
r = 0.02

result_ba, model_ba = solve_multiperiod_acp_bid_ask(
    quotes=my_bid_ask_quotes,
    S0=S0,
    r=r,
    ell_lower=1e-10,
    quote_floor=0.0,
)

print_bid_ask_acp_report(result_ba, model_ba)

# -------------------------
# Frictionless ACP custom template
# -------------------------

my_mid_quotes = mid_quotes_from_bid_ask(my_bid_ask_quotes)

result_fr, model_fr = solve_multiperiod_acp_frictionless(
    quotes=my_mid_quotes,
    S0=S0,
    r=r,
    ell_lower=1e-10,
)

print_frictionless_acp_report(result_fr, model_fr)
